In [1]:
## Assign 2 

## Asignment 2

In [2]:
# Setting myself up to work with the data

In [3]:
# Check my directory location and contents to ensure I can access the data files.
import os
os.getcwd()

'c:\\Users\\miche\\IFN647 Workshop Folder\\Assignment 2'

In [4]:
## Load the data 
folder = "Doc_Collection"
print(os.listdir(folder))

['.DS_Store', 'Dataset101', 'Dataset102', 'Dataset103', 'Dataset104', 'Dataset105', 'Dataset106', 'Dataset107', 'Dataset108', 'Dataset109', 'Dataset110', 'Dataset111', 'Dataset112', 'Dataset113', 'Dataset114', 'Dataset115', 'Dataset116', 'Dataset117', 'Dataset118', 'Dataset119', 'Dataset120', 'Dataset121', 'Dataset122', 'Dataset123', 'Dataset124', 'Dataset125', 'Dataset126', 'Dataset127', 'Dataset128', 'Dataset129', 'Dataset130', 'Dataset131', 'Dataset132', 'Dataset133', 'Dataset134', 'Dataset135', 'Dataset136', 'Dataset137', 'Dataset138', 'Dataset139', 'Dataset140', 'Dataset141', 'Dataset142', 'Dataset143', 'Dataset144', 'Dataset145', 'Dataset146', 'Dataset147', 'Dataset148', 'Dataset149', 'Dataset150']


In [5]:
import os
print(os.listdir("Doc_Collection"))


['.DS_Store', 'Dataset101', 'Dataset102', 'Dataset103', 'Dataset104', 'Dataset105', 'Dataset106', 'Dataset107', 'Dataset108', 'Dataset109', 'Dataset110', 'Dataset111', 'Dataset112', 'Dataset113', 'Dataset114', 'Dataset115', 'Dataset116', 'Dataset117', 'Dataset118', 'Dataset119', 'Dataset120', 'Dataset121', 'Dataset122', 'Dataset123', 'Dataset124', 'Dataset125', 'Dataset126', 'Dataset127', 'Dataset128', 'Dataset129', 'Dataset130', 'Dataset131', 'Dataset132', 'Dataset133', 'Dataset134', 'Dataset135', 'Dataset136', 'Dataset137', 'Dataset138', 'Dataset139', 'Dataset140', 'Dataset141', 'Dataset142', 'Dataset143', 'Dataset144', 'Dataset145', 'Dataset146', 'Dataset147', 'Dataset148', 'Dataset149', 'Dataset150']


In [6]:
print(os.listdir("Doc_Collection/Dataset101"))


['18586.xml', '22170.xml', '22513.xml', '26642.xml', '26847.xml', '27577.xml', '30647.xml', '39496.xml', '46547.xml', '46974.xml', '61329.xml', '6146.xml', '61780.xml', '62325.xml', '63261.xml', '77909.xml', '80425.xml', '80950.xml', '81463.xml', '82330.xml', '82454.xml', '82912.xml', '83167.xml']


## BM25 MODEL

In [7]:
import os, glob

base = "Doc_Collection"
print("Doc_Collection contents:", os.listdir(base))

ds = os.path.join(base, "Dataset101")
print("Dataset101 exists:", os.path.isdir(ds))
print("Dataset101 contents:", os.listdir(ds))

print("All files with any extension:", glob.glob(ds + "/*"))

Doc_Collection contents: ['.DS_Store', 'Dataset101', 'Dataset102', 'Dataset103', 'Dataset104', 'Dataset105', 'Dataset106', 'Dataset107', 'Dataset108', 'Dataset109', 'Dataset110', 'Dataset111', 'Dataset112', 'Dataset113', 'Dataset114', 'Dataset115', 'Dataset116', 'Dataset117', 'Dataset118', 'Dataset119', 'Dataset120', 'Dataset121', 'Dataset122', 'Dataset123', 'Dataset124', 'Dataset125', 'Dataset126', 'Dataset127', 'Dataset128', 'Dataset129', 'Dataset130', 'Dataset131', 'Dataset132', 'Dataset133', 'Dataset134', 'Dataset135', 'Dataset136', 'Dataset137', 'Dataset138', 'Dataset139', 'Dataset140', 'Dataset141', 'Dataset142', 'Dataset143', 'Dataset144', 'Dataset145', 'Dataset146', 'Dataset147', 'Dataset148', 'Dataset149', 'Dataset150']
Dataset101 exists: True
Dataset101 contents: ['18586.xml', '22170.xml', '22513.xml', '26642.xml', '26847.xml', '27577.xml', '30647.xml', '39496.xml', '46547.xml', '46974.xml', '61329.xml', '6146.xml', '61780.xml', '62325.xml', '63261.xml', '77909.xml', '80425.x

In [8]:
import math
import glob
import os

# ---------------------------------------------------------
# Parse query title into term frequencies
# ---------------------------------------------------------
def parse_query(title):
    title = title.lower()
    tokens = title.split()
    Qi = {}
    for t in tokens:
        Qi[t] = Qi.get(t, 0) + 1   # FIXED: was q.get()
    return Qi

# ---------------------------------------------------------
# Parse XML documents in a dataset folder
# ---------------------------------------------------------
def parse_documents(dataset_path):
    docs = {}

    for file in glob.glob(dataset_path + "/*.xml"):
        with open(file, "r", encoding="latin-1", errors="ignore") as f:
            content = f.read()

        lower = content.lower()

        # 1 Extract doc_id from itemid="..."
        doc_id = None
        if 'itemid="' in lower:
            doc_id = lower.split('itemid="')[1].split('"')[0].strip()
        if not doc_id:
            continue

        # 2 Extract text inside <text>...</text>
        if "<text>" in lower and "</text>" in lower:
            text_block = lower.split("<text>")[1].split("</text>")[0]
        else:
            continue

        # 3 Remove paragraph tags
        text_block = text_block.replace("<p>", " ").replace("</p>", " ")

        # 4 Tokenize and count term frequencies
        tokens = text_block.split()
        length = len(tokens)
        term_freqs = {}
        for t in tokens:
            term_freqs[t] = term_freqs.get(t, 0) + 1

        docs[doc_id] = {"terms": term_freqs, "length": length}

    return docs

# ---------------------------------------------------------
# Compute N and avdl
# ---------------------------------------------------------
def compute_collection_stats(docs):
    N = len(docs)
    if N == 0:
        return 0, 0
    total_length = sum(docs[d]["length"] for d in docs)
    avdl = total_length / N
    return N, avdl

# ---------------------------------------------------------
# Compute nt for each query term
# ---------------------------------------------------------
def compute_nt(Qi, docs):
    nt = {}
    for t in Qi:
        count = 0
        for d in docs:
            if t in docs[d]["terms"]:
                count += 1
        nt[t] = count
    return nt

# ---------------------------------------------------------
# BM25 scoring
# ---------------------------------------------------------
def bm25_score(Qi, docs, N, avdl, nt):
    k1 = 1.2
    k2 = 500
    b = 0.75

    Score = {d: 0 for d in docs}

    for t, qft in Qi.items():
        if nt[t] == 0:
            continue

        for d in docs:
            ft = docs[d]["terms"].get(t, 0)
            dl = docs[d]["length"]
            K = k1 * ((1 - b) + b * (dl / avdl))

            part1 = math.log10((N - nt[t] + 0.5) / (nt[t] + 0.5))
            part2 = ((k1 + 1) * ft) / (K + ft) if ft > 0 else 0
            part3 = ((k2 + 1) * qft) / (k2 + qft)

            Score[d] += part1 * part2 * part3

    return Score

# ---------------------------------------------------------
# Load Topics.txt
# ---------------------------------------------------------
def load_topics(path):
    topics = {}
    current_id = None
    current_title = None

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            stripped = line.strip()

            if stripped.startswith("<num>"):
                parts = stripped.replace("<num>", "").replace("Number:", "").strip().split()
                current_id = parts[-1]

            elif stripped.startswith("<title>"):
                title = stripped.replace("<title>", "").strip()
                current_title = title

                if current_id and current_title:
                    topics[current_id] = current_title
                    current_id = None
                    current_title = None

    return topics

# ---------------------------------------------------------
# MAIN PROGRAM
# ---------------------------------------------------------
if __name__ == "__main__":

    topics = load_topics("Topics.txt")

    for Ri, title in topics.items():

        Qi = parse_query(title)

        dataset_number = Ri[1:]
        dataset_path = f"Doc_Collection/Dataset{dataset_number}"

        docs = parse_documents(dataset_path)

        N, avdl = compute_collection_stats(docs)
        if N == 0:
            print("No valid documents for", Ri)
            continue

        nt = compute_nt(Qi, docs)
        Score = bm25_score(Qi, docs, N, avdl, nt)

        ranked = sorted(Score.items(), key=lambda x: x[1], reverse=True)

        # PRINT TO PYTHON OUTPUT
        print(f"{Ri} (Doc_ID BM25_Score):")
        for doc_id, score in ranked:
            print(f"{doc_id} {score}")
        print("...")

        # SAVE TO FILE
        os.makedirs("ModelOutputs", exist_ok=True)
        outname = f"ModelOutputs/Baseline1_{Ri}_Ranking.dat"
        with open(outname, "w") as f:
            f.write(f"{Ri} (Doc_ID BM25_Score):\n")
            for doc_id, score in ranked:
                f.write(f"{doc_id} {score}\n")
            f.write("...\n")

        print("Saved:", outname)

R101 (Doc_ID BM25_Score):
46547 1.8141868798492051
46974 1.8141868798492051
61329 0.9250550396681747
6146 0.7026770041387415
62325 0.6919779133070593
22513 0.6412094095593788
22170 0.6192099292361417
61780 0.5649435341946595
18586 0.0
26642 0.0
26847 0.0
27577 0.0
30647 0.0
39496 0.0
63261 0.0
77909 0.0
80425 0.0
80950 0.0
81463 0.0
82330 0.0
82454 0.0
82912 0.0
83167 0.0
...
Saved: ModelOutputs/Baseline1_R101_Ranking.dat
R102 (Doc_ID BM25_Score):
26061 2.8335165688857
58476 2.748983541015053
76635 2.54712406191787
78836 2.4922297438346623
12769 2.361076179877436
12767 2.2933525246151545
82227 1.5177858220741822
65414 1.3992892113219773
65036 1.304788577091841
65446 1.2932961533379452
73038 1.205873450396004
64645 1.1743459024763196
9479 1.1259752849123232
86929 1.1002426767513531
86912 1.099899048414823
32366 0.9555848401431691
26603 0.9443106726451855
57914 0.9017544083964414
25096 0.8082137156129247
14713 0.7727454208555101
11922 0.7581068992443782
3972 0.7173400531872888
26611 0.70

In [9]:
docs = parse_documents("Doc_Collection/Dataset101")
example_id = list(docs.keys())[0]
print("Document ID:", example_id)
print("Term frequencies:", docs[example_id]["terms"])
print("Document length:", docs[example_id]["length"])


Document ID: 18586
Term frequencies: {"germany's": 1, 'volkswagen': 2, 'ag': 1, ',': 1, 'unveiling': 1, 'a': 14, 'new': 6, 'passat': 5, 'car': 1, 'aimed': 1, 'at': 8, 'boosting': 1, 'its': 7, 'share': 1, 'of': 13, 'the': 50, 'mid-size': 3, 'market,': 1, 'said': 9, 'on': 9, 'wednesday': 2, 'total': 1, 'sales': 4, 'for': 13, 'first': 3, 'seven': 1, 'months': 1, '1996': 2, 'were': 4, 'up': 4, 'in': 21, 'europe': 4, 'and': 8, 'world.': 1, 'wolfsburg-based': 1, 'carmaker': 4, 'worldwide': 1, 'deliveries': 2, 'to': 21, 'customers': 1, 'from': 3, 'january': 1, 'july': 1, '13.5': 1, 'percent': 4, '2.35': 1, 'million': 4, 'vehicles,': 1, 'while': 1, 'western': 2, '11.3': 1, '1.451': 1, 'vehicles.': 1, 'but': 6, 'german': 5, 'market': 4, "europe's": 1, 'largest': 1, 'was': 6, 'still': 1, 'anaemic,': 1, 'with': 5, 'growth': 1, 'only': 2, '2.8': 1, 'period.': 1, 'vw': 13, 'management': 2, 'board': 2, 'chairman': 1, 'ferdinand': 1, 'piech': 5, 'late': 1, 'tuesday': 1, 'news': 1, 'conference': 1, 'u

In [10]:
print("Loaded topics:", len(topics))
print(topics)


Loaded topics: 50
{'R101': 'Economic espionage', 'R102': 'Convicts, repeat offenders', 'R103': 'Ferry Boat sinkings', 'R104': 'Rescue of kidnapped children', 'R105': 'Sport Utility Vehicles U.S.', 'R106': 'Government supported school vouchers', 'R107': 'Tourism Great Britain', 'R108': 'Harmful weight-loss drugs', 'R109': 'Child custody cases', 'R110': 'Terrorism Middle East tourism', 'R111': 'Telemarketing practices U.S.', 'R112': 'School bus accidents', 'R113': 'Ford foreign ventures', 'R114': 'Effects of global warming', 'R115': 'Indian casino laws', 'R116': 'Archaeology discoveries', 'R117': 'Organ transplants in the UK', 'R118': 'Progress in treatment of schizophrenia', 'R119': 'U.S. gas prices', 'R120': 'Deaths mining accidents', 'R121': 'China Pakistan nuclear missile', 'R122': "Symptoms Parkinson's disease", 'R123': 'Newspaper circulation decline', 'R124': 'Aborigine health', 'R125': 'Scottish Independence', 'R126': 'Nuclear plants U.S.', 'R127': 'U.S. automobile seat belt', 'R1

## JM Smoothing Model 

In [11]:
def jm_score(Qi, docs, lambda_val=0.3):
    # |DatasetRi| = total number of word occurrences
    DatasetRi = sum(docs[d]["length"] for d in docs)

    # cqj = total frequency of qj across DatasetRi
    cqj = {}
    for d in docs:
        for term, freq in docs[d]["terms"].items():
            cqj[term] = cqj.get(term, 0) + freq

    Score = {}
    for DocID in docs:
        D = docs[DocID]["length"]   # document length
        Score[DocID] = 0.0

        for qj in Qi:  # each query term
            fqj_D = docs[DocID]["terms"].get(qj, 0)  # fqj,D

            # JM smoothing formula
            p = (1 - lambda_val) * (fqj_D / float(D)) \
                + lambda_val * (cqj.get(qj, 0) / float(DatasetRi))

            if p > 0:
                Score[DocID] += math.log10(p)

    return Score

In [12]:
for Ri, title in topics.items():

    # Parse query into Qi
    Qi = parse_query(title)

    dataset_number = Ri[1:]
    dataset_path = f"Doc_Collection/Dataset{dataset_number}"

    docs = parse_documents(dataset_path)

    if len(docs) == 0:
        continue

    # JM smoothing scoring
    Score = jm_score(Qi, docs, lambda_val=0.3)
    ranked = sorted(Score.items(), key=lambda x: x[1], reverse=True)

    # PRINT TO PYTHON OUTPUT
    print(f"Example ranking in Baseline2_{Ri}_Ranking.dat:")
    print(f"Query is the topic title = \"{title}\"\n")
    print("Doc_ID\tJM_Score")
    for DocID, score in ranked:
        print(f"{DocID}\t{score}")
    print("...")

    # SAVE TO FILE
    os.makedirs("ModelOutputs", exist_ok=True)
    outname = f"ModelOutputs/Baseline2_{Ri}_Ranking.dat"

    with open(outname, "w") as f:
        f.write(f"Example ranking in Baseline2_{Ri}_Ranking.dat:\n")
        f.write(f"Query is the topic title = \"{title}\"\n\n")
        f.write("Doc_ID\tJM_Score\n")
        for DocID, score in ranked:
            f.write(f"{DocID}\t{score}\n")
        f.write("...\n")

    print("Saved:", outname)

Example ranking in Baseline2_R101_Ranking.dat:
Query is the topic title = "Economic espionage"

Doc_ID	JM_Score
46547	-3.8313063942329135
46974	-3.8313063942329135
61329	-5.413474114321531
6146	-5.652173668907822
62325	-5.68063732026487
22513	-5.8385295673222135
22170	-5.937090553610193
61780	-6.054466965989125
18586	-6.823155897336662
26642	-6.823155897336662
26847	-6.823155897336662
27577	-6.823155897336662
30647	-6.823155897336662
39496	-6.823155897336662
63261	-6.823155897336662
77909	-6.823155897336662
80425	-6.823155897336662
80950	-6.823155897336662
81463	-6.823155897336662
82330	-6.823155897336662
82454	-6.823155897336662
82912	-6.823155897336662
83167	-6.823155897336662
...
Saved: ModelOutputs/Baseline2_R101_Ranking.dat
Example ranking in Baseline2_R102_Ranking.dat:
Query is the topic title = "Convicts, repeat offenders"

Doc_ID	JM_Score
26061	-4.8728175997934535
76635	-4.878475736608632
58476	-4.971925986533249
78836	-5.0350951095015315
12769	-5.170193692466722
12767	-5.24640